# Web Scraping
---
In today's lab, we are going to download data from the internet using an API. API stands for **a**pplication **p**rogramming **i**nterface. Companies often create APIs as a way to allow users to more directly interact with their servers to retrieve data. Today, we are going to be using Twitter's API to download tweets to get some experience with large data.

In [1]:
# Run this cell to set up your notebook
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import warnings
import twitter_utils as tu

# Ensure that Pandas shows at least 280 characters in columns, so we can see full tweets
pd.set_option('max_colwidth', 280)

%matplotlib inline
import seaborn as sns
sns.set()
sns.set_context("talk")
import re
import json

## Setup
---
For this lab, we will be importing utility functions to interact with Twitter's API. Underneath the hood, these utility functions use the `tweepy` package, which is the how you can interface with Twitter's API using `python`. First, we need to install `tweepy` so that our utility functions will be able to use use it. If you have time at the end of this lab, you can look in the `twitter_utils.py` file in this folder and try to understand how the utility functions work.

In [2]:
!pip install tweepy

Twitter requires you to have authentication keys to access their API.  To get your keys, you'll have to sign up for a Twitter developer account. Note that **anyone who has your authentication keys can post as you**. In order to protect your keys, you will be storing them in a separate file, which we have called `keys.json`, and reading them into this notebook from that file. Also note that **Twitter limits developers to a certain rate of data requests**. This means that if you make too many API calls in a short period of time, Twitter may block you from retrieving data for a certain period of time. Avoid rerunning cells that retrieve new tweets.

Follow the instructions below to get your Twitter API keys.  **Read the instructions completely before starting.**

1. [Create a Twitter account](https://twitter.com).  You can use an existing account if you have one; if you prefer to not do this assignment under your regular account, feel free to create a throw-away account.
2. Under account settings, add your phone number to the account.
3. [Create a Twitter developer account](https://dev.twitter.com/resources/signup) by clicking the 'Apply' button on the top right of the page. Attach it to your Twitter account. You'll have to fill out a form describing what you want to do with the developer account. Explain that you are doing this for a class at UC Berkeley and that you don't know exactly what you're building yet and just need the account to get started. These applications are approved by some sort of AI system, so it doesn't matter exactly what you write.
4. Once you're logged into your developer account, [create an application for this assignment](https://apps.twitter.com/app/new).  You can call it whatever you want, and you can write any URL when it asks for a web site.  You don't need to provide a callback URL.
5. On the page for that application, find your Consumer Key and Consumer Secret.
6. On the same page, create an Access Token.  Record the resulting Access Token and Access Token Secret.
7. Edit the file `keys.json` in the same folder as this file and replace the placeholders with your keys.

Now you should be all ready to go! Let's test that you have correctly set up your developer account and the `keys.json` folder. The following cell loads your keys into this notebook, then validates them with the Twitter API. It should display your Twitter username without any warnings.

In [3]:
import json
key_file = "monicas_keys.json" #Change to "keys.json" in final version.

# Loading your keys from keys.json
with open(key_file) as f:
    keys = json.load(f)
# if you print or view the contents of keys be sure to delete the cell!

# Validate keys
tu.validate_authentication(keys)

The keys are valid. Your username is: data100mw


If you are getting any errors in this cell, ask a TA for help. If you do not have valid keys, you will not be able to use the API.

## Downloading Tweets
---
Now we should be ready to download some tweets! In the following cell, we use one of the utility functions to download recent tweets with the hashtag "#data".

In [4]:
# Note that you do not write the actual hashtag symbol
data = tu.download_recent_tweets_by_hashtag(hashtag = "data",
                                           keys = keys)

NameError: name 'TweepError' is not defined

We now have `data` assigned to a list with each element corresponding to a tweet. Let's examine one of these elements to get a better understanding of our data.

In [ ]:
data[0]

This is an example of another `python` data structure called a *dictionary*. Dictionaries store values by associating them with a *key* rather than by an integer index. You can index into a dictionary using bracket notation just like a list. For example

In [ ]:
d = {'a': 1,
    'b': 2,
    'c': 3}
d['a']

### Data Cleaning
---
The dictionary we were looking at above is a little bit hard to interpret because there dictionaries nested inside of some our keys. We can look only at the first level of keys in our dictionary by using the `.keys()` method.

In [ ]:
data[0].keys()

The most relevant keys for what we will be doing today are `'text'` (which contains the body of the text), `'user'` (the user who posted the tweet),  `'created_at'` (which is the time that the tweet was posted), and `'retweet_count'` (which is the number of times a tweet was retweeted). We can convert this into a dataframe by simply passing it to the `pd.DataFrame()` function.

In [ ]:
df = pd.DataFrame(data)

df = df[['text', 'user', 'created_at', 'retweet_count']]

Unfortunately, Twitter by default does not attach geographic data to the metadata of each tweet. To get around this, we can use the location associated to the account of each poster. The code for the folloiwng section was adapted from this [source](http://www.mikaelbrunila.fi/2017/03/27/scraping-extracting-mapping-geodata-twitter/).

In [ ]:
tweets_and_locations_list = list()
for tweet in data:
    new_entry = {}
    new_entry['tweet'] = tweet['text']
    new_entry['location'] = tweet['user']['location']
    tweets_and_locations_list.append(new_entry)
    
tweets_and_locations = pd.DataFrame(tweets_and_locations)
tweets_and_locations.head(10)

Clearly this isn't a foolproof method, since the location associated with an account may have little bearing on the actual location from which a tweet was posted. Also, not all users have a location connected to their account. Depending on the data you have pulled from Twitter, you may also notice that some of the "locations" are not actually real places. We can do a bit of data cleaning to filter out the rows that contain true locations. First, let's get rid of the rows that do not contain any text at all in the `location` column.

In [ ]:
no_empties = pd.DataFrame(columns = ['location', 'tweet'])
for i in range(len(tweets_and_locations)):
    if tweets_and_locations.loc[i, "location"] != '':
        no_empties = no_empties.append(tweets_and_locations.loc[i,:])
no_empties.head(10)

This looks pretty good! We would still like to filter through our locations for places that actually exist. Let's use the `.groupby()` method to take a look at what locations we have in our data.

In [ ]:
no_empties.groupby('location').count()

If you scroll through this list, you will likely see a whole litany of "locations" that do not resemble locations, from things like phone numbers, to IP addresses, to emojis. You may even see locations in other languages! We do not have time to day to sort through all of these right now, so we are goign to move on to a few other techniques that we can use to analyze these kinds of data.

### Temporal Data
---
Another facet of the tweets that you may want to analyze is the time at which they were posted. Currently, the only way we have information about the time the tweets were posted is in the `'created_at'` column, which is a string. As you may remember from the Introductory lab, `python` compares strings by assigning values to the letters themselves based on their position in the alphabet. We want to convert these strings to `datetime` objects, which will tell `python` at what time tweets were posted.

In [ ]:
df['time'] = pd.to_datetime(df['created_at'])
df['time'].head()

Now that each string has been converted into a `datetime` object, we can extract the day, hour, minute, etc. of each time point like so

In [ ]:
df.loc[0, 'time'].day

In [ ]:
df.loc[0, 'time'].hour

In [ ]:
df.loc[0, 'time'].minute

Notice that we are not adding parentheses at the end of each line. That is because the `.day`, `.hour`, and `.minute` are not *functions* we are calling, but rather *attributes* of the particular `datetime` object. If we want to look at the time of day that people tend to tweet about #data, we can extract these attributes.

In [ ]:
df['hour'] = [df.loc[i, 'time'].hour + df.loc[i, 'time'].minute/60 + df.loc[i, 'time'].second/3600 for i in range(len(df))]
df['hour'].hist()
plt.xlabel("Hour (UTC)")
plt.ylabel("Number of Tweets");

**Question:** What observations or trends do you notice about this graph?

YOUR ANSWER HERE

**Question:** What could be improved about this graph or the process we used to obtain the data that generated it?

YOUR ANSWER HERE